# Parallel Optimization and Performance Analysis

This notebook demonstrates parallel optimization capabilities and performance analysis using `ws3`.

> **Prerequisites**: Completion of `070_ws3_quickstart_complete_workflow.ipynb`

## What You'll Learn

- How to use multi-core parallelization
- How to benchmark solver performance
- How to optimize parallel parameters
- How to scale to large problems

In [ ]:
%load_ext autoreload
%autoreload 2

import time
import os
import multiprocessing
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import ws3.forest
import ws3.opt
from util import compile_scenario, plot_scenario

In [ ]:
# Check available CPU cores
num_cores = multiprocessing.cpu_count()
print(f"Available CPU cores: {num_cores}")
print(f"OS: {os.uname().sysname}")

In [ ]:
# Model parameters
base_year = 2020
horizon = 20  # longer horizon for parallel testing
period_length = 10
max_age = 1000
tvy_name = "totvol"

print(f"Model Parameters:")
print(f"  Base Year: {base_year}")
print(f"  Horizon: {horizon} periods")
print(f"  Period Length: {period_length} years")

In [ ]:
# Create ForestModel
fm = ws3.forest.ForestModel(
    model_name="tsa24",
    model_path="data/woodstock_model_files_tsa24",
    base_year=base_year,
    horizon=horizon,
    period_length=period_length,
    max_age=max_age
)
fm.import_landscape_section()
fm.import_areas_section(convert_periods_to_years=period_length)
fm.import_yields_section(convert_periods_to_years=period_length)
fm.import_actions_section(convert_periods_to_years=period_length)
fm.import_transitions_section(convert_periods_to_years=period_length)
fm.initialize_areas()
fm.add_null_action()
fm.reset_actions()
fm.actions["harvest"].is_harvest = True

print(f"ForestModel loaded: {fm}")

In [ ]:
# Define coefficient functions
import functools

expr = "0.85 * totvol"

coeff_funcs = {
    "z": functools.partial(ws3.opt.cmp_c_z, expr=expr),
    "cflw_hv": functools.partial(ws3.opt.cmp_c_caa, expr=expr, acodes=["harvest"]),
    "cflw_ha": functools.partial(ws3.opt.cmp_c_caa, expr="1.", acodes=["harvest"]),
    "cgen_gs": functools.partial(ws3.opt.cmp_c_ci, yname=tvy_name, mask=None)
}

cflw_e = {
    "cflw_hv": ({p: 0.05 for p in fm.periods}, 1),
    "cflw_ha": ({p: 0.05 for p in fm.periods}, 1)
}

gs_lb_rhs = fm.inventory(0, "totvol") * 0.90
cgen_data = {
    "cgen_gs": {"lb": {10: gs_lb_rhs}, "ub": {10: 999999999.}}
}

acodes = ("null", "harvest")

print("Coefficient functions and constraints defined")

In [ ]:
# Benchmark with different worker counts
worker_counts = [1, 2, 4, 8, 16]
build_times = []
solve_times = []
total_times = []

print("Benchmarking parallel performance...\n")

for workers in worker_counts:
    print(f"Testing with {workers} worker(s)...")
    
    # Build problem
    t0 = time.perf_counter()
    problem = fm.add_problem(
        name=f"benchmark_{workers}",
        coeff_funcs=coeff_funcs,
        cflw_e=cflw_e,
        cgen_data=cgen_data,
        acodes=acodes,
        sense=ws3.opt.SENSE_MAXIMIZE,
        mask=None,
        workers=workers,
        verbose=False
    )
    t1 = time.perf_counter()
    build_time = t1 - t0
    
    # Solve problem
    t2 = time.perf_counter()
    problem.solve(verbose=False, threads=workers)
    t3 = time.perf_counter()
    solve_time = t3 - t2
    
    total_time = build_time + solve_time
    
    build_times.append(build_time)
    solve_times.append(solve_time)
    total_times.append(total_time)
    
    print(f"  Build: {build_time:.2f}s, Solve: {solve_time:.2f}s, Total: {total_time:.2f}s")

print("\nBenchmarking complete!")

In [ ]:
# Visualize performance results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Build time
axes[0].plot(worker_counts, build_times, 'o-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Workers')
axes[0].set_ylabel('Build Time (s)')
axes[0].set_title('Problem Build Time')
axes[0].grid(True, alpha=0.3)

# Solve time
axes[1].plot(worker_counts, solve_times, 's-', linewidth=2, markersize=8, color='orange')
axes[1].set_xlabel('Number of Workers')
axes[1].set_ylabel('Solve Time (s)')
axes[1].set_title('Solver Time')
axes[1].grid(True, alpha=0.3)

# Total time
axes[2].plot(worker_counts, total_times, '^-', linewidth=2, markersize=8, color='green')
axes[2].set_xlabel('Number of Workers')
axes[2].set_ylabel('Total Time (s)')
axes[2].set_title('Total Time (Build + Solve)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Calculate speedup and efficiency
speedups = [total_times[0] / t for t in total_times]
efficiencies = [s / w for s, w in zip(speedups, worker_counts)]

print("Performance Metrics:")
print("=" * 60)
print(f"{'Workers':<10} {'Total Time':<15} {'Speedup':<15} {'Efficiency':<15}")
print("-" * 60)
for i, workers in enumerate(worker_counts):
    print(f"{workers:<10} {total_times[i]:<15.2f} {speedups[i]:<15.2f} {efficiencies[i]:<15.2f}")
print("=" * 60)

In [ ]:
# Visualize speedup and efficiency
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Speedup
axes[0].plot(worker_counts, speedups, 'o-', linewidth=2, markersize=8, color='blue')
axes[0].axhline(y=1, color='r', linestyle='--', alpha=0.5, label='No speedup')
axes[0].axhline(y=max(worker_counts), color='g', linestyle='--', alpha=0.5, label='Ideal speedup')
axes[0].set_xlabel('Number of Workers')
axes[0].set_ylabel('Speedup (relative to 1 worker)')
axes[0].set_title('Parallel Speedup')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Efficiency
axes[1].plot(worker_counts, efficiencies, 's-', linewidth=2, markersize=8, color='green')
axes[1].axhline(y=1, color='r', linestyle='--', alpha=0.5, label='100% efficiency')
axes[1].set_xlabel('Number of Workers')
axes[1].set_ylabel('Efficiency (speedup / workers)')
axes[1].set_title('Parallel Efficiency')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Solve with optimal worker count and compile results
optimal_workers = worker_counts[np.argmin(total_times)]
print(f"Optimal worker count: {optimal_workers}")

# Solve with optimal workers
problem = fm.add_problem(
    name="optimal",
    coeff_funcs=coeff_funcs,
    cflw_e=cflw_e,
    cgen_data=cgen_data,
    acodes=acodes,
    sense=ws3.opt.SENSE_MAXIMIZE,
    mask=None,
    workers=optimal_workers,
    verbose=False
)

t0 = time.perf_counter()
problem.solve(verbose=False, threads=optimal_workers)
t1 = time.perf_counter()

print(f"Solve time with {optimal_workers} workers: {t1 - t0:.2f}s")

if problem.status() == ws3.opt.STATUS_OPTIMAL:
    sch = fm.compile_schedule(problem)
    fm.apply_schedule(sch,
                     force_integral_area=False,
                     override_operability=False,
                     fuzzy_age=False,
                     recourse_enabled=False,
                     verbose=False,
                     compile_c_ycomps=True)
    df = compile_scenario(fm)
    print(f"\nSolution compiled successfully")
    print(df.head())

In [ ]:
# Summary of parallel optimization
print("=" * 60)
print("PARALLEL OPTIMIZATION SUMMARY")
print("=" * 60)
print(f"Available Cores: {num_cores}")
print(f"Tested Worker Counts: {worker_counts}")
print(f"Optimal Workers: {optimal_workers}")
print(f"Best Total Time: {min(total_times):.2f}s")
print(f"Best Speedup: {max(speedups):.2f}x")
print(f"Best Efficiency: {max(efficiencies):.2f}%")
print("=" * 60)
print("\nRecommendations:")
print("  - Use optimal worker count for best performance")
print("  - Monitor diminishing returns with more workers")
print("  - Consider problem size when choosing workers")